# Stage 2 Full Pipeline Colab

Refactored notebook for C-MAPSS Stage 2 experiments. Shared helpers handle setup, experiment execution, summary collection, and final result aggregation so each stage only defines a compact experiment list.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import itertools
import json
import os
import subprocess

import pandas as pd

PROJECT_ROOT = Path('/content/ECE-228-project')
DRIVE_ROOT = Path('/content/drive/MyDrive/ece228_stage2')
REPO_URL = 'https://github.com/m8wei-coder/ECE-228-project.git'

def shell(cmd, cwd=None):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if PROJECT_ROOT.exists():
    shell('git pull --rebase', cwd=PROJECT_ROOT)
else:
    shell(f'git clone {REPO_URL} {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shell('pip -q install -r stage2_refactor/requirements.txt')
shell('python3 stage2_refactor/tools/smoke_read_data.py --data-dir CMaps')

In [ ]:
import torch, pandas, sklearn, scipy
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

## 2. Shared Helpers

In [ ]:
def ftag(value):
    return str(value).replace('.', 'p').replace('-', 'm')

def default_run_id(exp, prefix=None):
    prefix = prefix or exp.get('subset', 'FD001').lower()
    return (
        f"{prefix}_{exp['model']}_"
        f"h{exp['hidden_size']}_l{exp['num_layers']}_"
        f"d{ftag(exp['dropout'])}_lr{ftag(exp['lr'])}_seed{exp.get('seed', 42)}"
    )

def run_experiment(exp, checkpoint_dir, output_dir, validation=True, resume=True):
    run_id = exp.get('run_id') or default_run_id(exp, exp.get('prefix'))
    cmd = [
        'python', '-m', 'stage2_refactor.experiments.run_experiment',
        '--subset', exp.get('subset', 'FD001'),
        '--model', exp['model'],
        '--run-id', run_id,
        '--mode', exp.get('mode', 'train_eval'),
        '--checkpoint-dir', str(checkpoint_dir),
        '--output-dir', str(output_dir),
        '--hidden-size', str(exp['hidden_size']),
        '--num-layers', str(exp['num_layers']),
        '--dropout', str(exp['dropout']),
        '--learning-rate', str(exp['lr']),
        '--seed', str(exp.get('seed', 42)),
    ]
    if validation:
        cmd.extend([
            '--validation-enabled',
            '--validation-fraction', str(exp.get('validation_fraction', 0.15)),
            '--validation-patience', str(exp.get('validation_patience', 10)),
        ])
    if resume:
        cmd.append('--resume')
    print('\n' + '=' * 100)
    print(run_id)
    print(' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Run failed: {run_id}')
    return run_id

def run_batch(experiments, name, validation=True, resume=True):
    checkpoint_dir = DRIVE_ROOT / f'checkpoints_{name}'
    output_dir = DRIVE_ROOT / f'outputs_{name}'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'Total experiments: {len(experiments)}')
    for i, exp in enumerate(experiments, start=1):
        print(f'[{i}/{len(experiments)}]')
        run_experiment(exp, checkpoint_dir, output_dir, validation=validation, resume=resume)
    return output_dir

SUMMARY_COLUMNS = [
    'subset', 'model', 'run_id', 'seed', 'hidden_size', 'num_layers', 'dropout',
    'learning_rate', 'validation_enabled', 'test_rmse', 'test_score', 'best_metric',
    'best_epoch', 'parameter_count', 'total_train_seconds', 'path'
]

def collect_summaries(output_dir, csv_name):
    output_dir = Path(output_dir)
    rows = []
    for summary_path in output_dir.glob('*/*/*/summary.json'):
        with open(summary_path) as f:
            s = json.load(f)
        s['path'] = str(summary_path)
        rows.append({col: s.get(col) for col in SUMMARY_COLUMNS})
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['subset', 'test_rmse', 'test_score'], ascending=[True, True, True])
    csv_path = output_dir / csv_name
    df.to_csv(csv_path, index=False)
    display(df)
    print(csv_path)
    return df, csv_path

def summarize_by_config(df, csv_path=None):
    group_cols = ['subset', 'model', 'hidden_size', 'num_layers', 'dropout', 'learning_rate']
    summary = (
        df.groupby(group_cols)
          .agg(
              rmse_mean=('test_rmse', 'mean'),
              rmse_std=('test_rmse', 'std'),
              score_mean=('test_score', 'mean'),
              score_std=('test_score', 'std'),
              params=('parameter_count', 'first'),
              n=('test_rmse', 'count'),
          )
          .reset_index()
          .sort_values(['subset', 'rmse_mean', 'score_mean'])
    )
    if csv_path is not None:
        summary.to_csv(csv_path, index=False)
        print(csv_path)
    display(summary)
    return summary

def grid(model, subset='FD001', hidden_sizes=(60,), layers=(2,), dropouts=(0.1,), lrs=(0.0005,), seed=42, prefix=None):
    return [
        {
            'subset': subset,
            'model': model,
            'hidden_size': h,
            'num_layers': l,
            'dropout': d,
            'lr': lr,
            'seed': seed,
            'prefix': prefix,
        }
        for h, l, d, lr in itertools.product(hidden_sizes, layers, dropouts, lrs)
    ]

## 3. Optional Stage 1 Artifact Check

Run only if you want to verify the checked-in Stage 1 checkpoint.

In [ ]:
RUN_STAGE1_ARTIFACT_CHECK = False

if RUN_STAGE1_ARTIFACT_CHECK:
    shell('mkdir -p /content/stage1_logs')
    shell('tar -xf reproduce/logs.tar -C /content/stage1_logs')
    cmd = [
        'python', '-m', 'stage2_refactor.experiments.run_experiment',
        '--subset', 'FD001', '--model', 'lstm', '--mode', 'eval',
        '--model-path', '/content/stage1_logs/logs/cmapss_lstm_fd001.pth',
        '--preprocessing-artifact-path', '/content/stage1_logs/logs/cmapss_condition_scalers_fd001.gz',
        '--output-dir', str(DRIVE_ROOT / 'stage1_weight_eval_outputs'),
        '--checkpoint-dir', str(DRIVE_ROOT / 'stage1_weight_eval_checkpoints'),
    ]
    subprocess.run(cmd, text=True, check=True)

## 4. Focused FD001 Validation Sweep

In [ ]:
val_sweep = []
val_sweep += grid('bigru', hidden_sizes=[60, 90], layers=[2], dropouts=[0.1, 0.2], lrs=[0.001, 0.0005], prefix='fd001_val')
val_sweep += grid('gru', hidden_sizes=[90, 128], layers=[2], dropouts=[0.1, 0.2], lrs=[0.001, 0.0005], prefix='fd001_val')

val_output_dir = run_batch(val_sweep, 'val_sweep', validation=True, resume=True)
val_df, _ = collect_summaries(val_output_dir, 'fd001_val_sweep_summary.csv')

## 5. FD001 Finalists: Multi-Seed

In [ ]:
finalist_specs = [
    ('gru', 90, 2, 0.2, 0.0005),
    ('gru', 90, 2, 0.1, 0.001),
    ('gru', 128, 2, 0.2, 0.001),
    ('bigru', 60, 2, 0.1, 0.0005),
]
finalists = []
for model, h, l, d, lr in finalist_specs:
    for seed in [7, 42, 123]:
        finalists += grid(model, hidden_sizes=[h], layers=[l], dropouts=[d], lrs=[lr], seed=seed, prefix='fd001_finalist')

finalist_output_dir = run_batch(finalists, 'finalists', validation=True, resume=True)
finalist_df, finalist_runs_path = collect_summaries(finalist_output_dir, 'fd001_finalists_runs.csv')
finalist_summary = summarize_by_config(finalist_df, finalist_output_dir / 'fd001_finalists_summary.csv')

## 6. Transfer Check on FD002-FD004

In [ ]:
transfer_candidates = [
    ('gru', 90, 2, 0.2, 0.0005),
    ('bigru', 60, 2, 0.1, 0.0005),
]
transfer = []
for subset in ['FD002', 'FD003', 'FD004']:
    for model, h, l, d, lr in transfer_candidates:
        transfer += grid(model, subset=subset, hidden_sizes=[h], layers=[l], dropouts=[d], lrs=[lr], seed=42, prefix=f'{subset.lower()}_transfer')

transfer_output_dir = run_batch(transfer, 'transfer', validation=True, resume=True)
transfer_df, transfer_path = collect_summaries(transfer_output_dir, 'transfer_seed42_summary.csv')

## 7. Final Confirmation Seeds

In [ ]:
final_confirm = []
for seed in [7, 123]:
    final_confirm += grid('bigru', subset='FD002', hidden_sizes=[60], layers=[2], dropouts=[0.1], lrs=[0.0005], seed=seed, prefix='fd002_final')
    final_confirm += grid('gru', subset='FD003', hidden_sizes=[90], layers=[2], dropouts=[0.2], lrs=[0.0005], seed=seed, prefix='fd003_final')
    final_confirm += grid('bigru', subset='FD004', hidden_sizes=[60], layers=[2], dropouts=[0.1], lrs=[0.0005], seed=seed, prefix='fd004_final')

confirm_output_dir = run_batch(final_confirm, 'final_confirm', validation=True, resume=True)
confirm_df, confirm_path = collect_summaries(confirm_output_dir, 'final_confirm_runs.csv')

## 8. Build Final Stage 2 Tables

In [ ]:
fd001_runs = pd.read_csv(DRIVE_ROOT / 'outputs_finalists/fd001_finalists_runs.csv')
transfer_runs = pd.read_csv(DRIVE_ROOT / 'outputs_transfer/transfer_seed42_summary.csv')
confirm_runs = pd.read_csv(DRIVE_ROOT / 'outputs_final_confirm/final_confirm_runs.csv')

fd001_final = fd001_runs[
    (fd001_runs['model'] == 'gru') &
    (fd001_runs['hidden_size'] == 90) &
    (fd001_runs['num_layers'] == 2) &
    (fd001_runs['dropout'] == 0.2) &
    (fd001_runs['learning_rate'] == 0.0005)
].copy()
fd001_final['subset'] = 'FD001'

transfer_final = transfer_runs[
    ((transfer_runs['subset'] == 'FD002') & (transfer_runs['model'] == 'bigru')) |
    ((transfer_runs['subset'] == 'FD003') & (transfer_runs['model'] == 'gru')) |
    ((transfer_runs['subset'] == 'FD004') & (transfer_runs['model'] == 'bigru'))
].copy()

all_final = pd.concat([fd001_final, transfer_final, confirm_runs], ignore_index=True)
all_final = all_final.sort_values(['subset', 'seed'])
final_summary = summarize_by_config(all_final)

runs_path = DRIVE_ROOT / 'final_stage2_runs.csv'
summary_path = DRIVE_ROOT / 'final_stage2_summary.csv'
all_final.to_csv(runs_path, index=False)
final_summary.to_csv(summary_path, index=False)
display(all_final)
display(final_summary)
print(runs_path)
print(summary_path)